# LeetCode 719 — Find K-th Smallest Pair Distance bằng Python

- Link: https://leetcode.com/problems/find-k-th-smallest-pair-distance/
- Độ khó: **Hard**
- Chủ đề: Sắp xếp, Hàng đợi ưu tiên (heap), **Tìm kiếm nhị phân trên đáp án**, Hai con trỏ (two pointers)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestDistancePair(self, nums: List[int], k: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Có `n` số, ta xét **mọi cặp** `(i, j)` với `i < j` và khoảng cách `|nums[i] - nums[j]|`. Trong khoảng `n(n-1)/2` khoảng cách đó (tới ~`5·10^7` khi `n = 10^4`), hãy tìm khoảng cách **nhỏ thứ k**.

Liệt kê hết rồi sắp xếp thì quá chậm và tốn bộ nhớ. Bài này là ví dụ mẫu mực cho kỹ thuật **tìm kiếm nhị phân trên đáp án**: thay vì tìm trực tiếp "khoảng cách thứ k", ta hỏi một câu dễ hơn — *"có bao nhiêu cặp có khoảng cách ≤ d?"* — câu hỏi này trả lời được trong `O(n)` bằng hai con trỏ, và kết quả **tăng dần theo `d`**.

Sau notebook này bạn sẽ:

1. Viết brute force liệt kê mọi cặp làm trọng tài.
2. Dùng heap để "trộn" các dãy khoảng cách đã sắp xếp (giống trộn k danh sách).
3. Hiểu tìm kiếm nhị phân trên đáp án: hàm đếm đơn điệu, tìm `d` **nhỏ nhất** có `count(d) ≥ k`.
4. Đếm số cặp có khoảng cách ≤ d bằng `bisect` (`O(n log n)`) và bằng hai con trỏ (`O(n)`).
5. Kiểm chứng mọi cách bằng test ngẫu nhiên có seed và nhận diện các bài cùng họ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** **Khoảng cách** của cặp số nguyên `a` và `b` là `|a - b|`. Cho mảng số nguyên `nums` và số nguyên `k`. Trả về khoảng cách **nhỏ thứ `k`** trong số các khoảng cách `|nums[i] - nums[j]|` với mọi cặp chỉ số `0 <= i < j < nums.length`.

(Các khoảng cách được đếm **có lặp**: nhiều cặp cùng khoảng cách thì mỗi cặp tính một lần.)

**Ví dụ 1**

```text
Input:  nums = [1,3,1], k = 1
Output: 0
Giải thích: các cặp: (1,3) -> 2, (1,1) -> 0, (3,1) -> 2. Sắp xếp: [0, 2, 2]. Nhỏ thứ 1 là 0.
```

**Ví dụ 2**

```text
Input:  nums = [1,1,1], k = 2
Output: 0
```

**Ví dụ 3**

```text
Input:  nums = [1,6,1], k = 3
Output: 5
```

**Ràng buộc**

- `n == nums.length`, `2 <= n <= 10^4`
- `0 <= nums[i] <= 10^6`
- `1 <= k <= n * (n - 1) / 2`

In [1]:
from itertools import combinations

for nums, k in [([1, 3, 1], 1), ([1, 1, 1], 2), ([1, 6, 1], 3)]:
    dists = sorted(abs(a - b) for a, b in combinations(nums, 2))
    print(f"nums={nums}, k={k}: mọi khoảng cách {dists} -> thứ {k} là {dists[k - 1]}")

nums=[1, 3, 1], k=1: mọi khoảng cách [0, 2, 2] -> thứ 1 là 0
nums=[1, 1, 1], k=2: mọi khoảng cách [0, 0, 0] -> thứ 2 là 0
nums=[1, 6, 1], k=3: mọi khoảng cách [0, 5, 5] -> thứ 3 là 5


## 3. Kiến thức nền

### 3.1. `itertools.combinations` — mọi cặp `i < j`

`combinations(nums, 2)` sinh mọi cặp phần tử theo thứ tự chỉ số tăng, mỗi cặp đúng một lần: `n(n-1)/2` cặp.

In [2]:
print(list(combinations([1, 3, 1], 2)), "| số cặp với n=10^4:", 10**4 * (10**4 - 1) // 2)

[(1, 3), (1, 1), (3, 1)] | số cặp với n=10^4: 49995000


### 3.2. Sắp xếp làm khoảng cách dễ tính

Sau khi sắp xếp tăng dần, với `i < j` thì `|nums[j] - nums[i]| = nums[j] - nums[i]`. Hơn nữa, với `i` cố định, khoảng cách **tăng** khi `j` tăng — đây là chìa khoá cho cả heap lẫn hai con trỏ. Thứ tự ban đầu không quan trọng vì tập các cặp không đổi khi hoán vị mảng.

In [3]:
nums = sorted([1, 6, 1, 3])
print("đã sắp:", nums)
for i in range(len(nums)):
    print(f"i={i}: khoảng cách với j > i:", [nums[j] - nums[i] for j in range(i + 1, len(nums))])

đã sắp: [1, 1, 3, 6]
i=0: khoảng cách với j > i: [0, 2, 5]
i=1: khoảng cách với j > i: [2, 5]
i=2: khoảng cách với j > i: [3]
i=3: khoảng cách với j > i: []


### 3.3. `heapq` — min-heap

`heappush(h, x)` thêm, `heappop(h)` lấy phần tử nhỏ nhất, đều `O(log size)`. Tuple so sánh theo phần tử đầu tiên trước.

In [4]:
import heapq
h = []
for item in [(5, 0, 1), (0, 1, 2), (2, 0, 2)]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(0, 1, 2), (2, 0, 2), (5, 0, 1)]


### 3.4. `bisect_right` — đếm số phần tử ≤ x

Với mảng tăng, `bisect_right(a, x)` = số phần tử `≤ x`. Dùng để đếm "bao nhiêu `nums[j] ≤ nums[i] + d`".

In [5]:
from bisect import bisect_right
a = [1, 1, 3, 6, 6, 9]
print(bisect_right(a, 3), bisect_right(a, 6), bisect_right(a, 0))

3 5 0


### 3.5. Tìm kiếm nhị phân "giá trị nhỏ nhất thoả điều kiện"

Nếu điều kiện `P(d)` **đơn điệu** (sai, sai, …, sai, đúng, đúng, …) thì tìm `d` nhỏ nhất có `P(d)` đúng bằng mẫu:

```python
lo, hi = MIN, MAX          # biết chắc P(hi) đúng
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid    # mid có thể là đáp án, giữ lại
    else:      lo = mid + 1
return lo
```

In [6]:
def first_true(lo, hi, P):
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print("số nguyên nhỏ nhất có bình phương >= 50:", first_true(0, 50, lambda x: x * x >= 50))

số nguyên nhỏ nhất có bình phương >= 50: 8


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Bảng khoảng cách (tính tay)

`nums = [1, 6, 1]` → sắp xếp `[1, 1, 6]`:

| | j=1 (1) | j=2 (6) |
|---|---|---|
| i=0 (1) | 0 | 5 |
| i=1 (1) | | 5 |

Các khoảng cách: `0, 5, 5` → thứ 3 là **5**.

### 4.2. Hàm đếm `count(d)` và vì sao nó giúp được

Đặt `count(d)` = số cặp có khoảng cách `≤ d`. Với `[1, 1, 6]`: `count(0) = 1`, `count(1..4) = 1`, `count(5) = 3`.

`count` **không giảm** theo `d`. Khoảng cách thứ `k` chính là **`d` nhỏ nhất có `count(d) ≥ k`**:

- `k = 1`: `count(0) = 1 ≥ 1` → đáp án 0.
- `k = 2`: `count(0..4) = 1 < 2`, `count(5) = 3 ≥ 2` → đáp án 5.
- `k = 3`: tương tự → 5.

Trực giác: "khoảng cách thứ `k` là `D`" ⇔ "có ít nhất `k` cặp `≤ D`, nhưng ít hơn `k` cặp `≤ D - 1`".

In [7]:
def count_pairs_naive(nums, d):
    return sum(1 for a, b in combinations(nums, 2) if abs(a - b) <= d)

nums = [1, 6, 1]
print([(d, count_pairs_naive(nums, d)) for d in range(0, 7)])

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 3), (6, 3)]


## 5. Cách 1 — Brute Force: liệt kê mọi khoảng cách rồi sắp xếp

In [8]:
def kth_brute(nums, k):
    dists = sorted(abs(nums[i] - nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))
    return dists[k - 1]

print(kth_brute([1, 3, 1], 1), kth_brute([1, 1, 1], 2), kth_brute([1, 6, 1], 3))

0 0 5


**Độ phức tạp.** `O(n² log n)` thời gian, `O(n²)` bộ nhớ. Với `n = 10^4`: `5·10^7` khoảng cách — list Python chiếm vài GB, sắp xếp mất hàng chục giây.

**Nút thắt.** Ta tạo ra **toàn bộ** khoảng cách chỉ để lấy **một** giá trị. Cần cách "đếm" thay vì "liệt kê".

(Một biến thể nhỏ: vì `nums[i] ≤ 10^6`, có thể dùng **mảng đếm** theo khoảng cách thay cho sắp xếp — `O(n² + W)` — vẫn bị chặn bởi `n²`.)

## 6. Bước đột phá

**Insight 1 — trộn các dãy đã sắp (heap).** Sau khi sắp `nums`, với mỗi `i`, dãy `nums[i+1]-nums[i], nums[i+2]-nums[i], …` **tăng dần**. Ta có `n - 1` dãy tăng; lấy phần tử nhỏ thứ `k` của hợp các dãy = trộn bằng heap, pop `k` lần. Tốt khi `k` nhỏ, nhưng `k` có thể tới `5·10^7`.

**Insight 2 — tìm kiếm nhị phân trên đáp án.** Đáp án nằm trong `[0, max - min]`. Điều kiện `P(d) = (count(d) ≥ k)` đơn điệu. Vậy:

> Tìm `d` nhỏ nhất có `count(d) ≥ k` bằng nhị phân → `O(log W)` lần gọi `count`, `W = max - min ≤ 10^6` (≈ 20 lần).

**Insight 3 — đếm nhanh bằng hai con trỏ.** Trên mảng đã sắp, với mỗi `right`, các `left` hợp lệ (`nums[right] - nums[left] ≤ d`) tạo thành một **đoạn liên tiếp** `[left_min, right - 1]`, và `left_min` **không giảm** khi `right` tăng. Vậy duyệt `right` từ trái sang phải, `left` chỉ tiến → `O(n)`:

> `count(d) = Σ_right (right - left)` với `left` là chỉ số nhỏ nhất có `nums[right] - nums[left] ≤ d`.

**Vì sao `lo` cuối cùng là một khoảng cách có thật?** `count(d)` chỉ "nhảy" tại các giá trị là khoảng cách của một cặp. `d*` nhỏ nhất có `count(d*) ≥ k` thì `count(d* - 1) < k ≤ count(d*)` → có cặp với khoảng cách **đúng bằng** `d*`.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Ý tưởng | Time |
|---|---|---|
| 1 | Liệt kê + sắp xếp | `O(n² log n)` |
| 2 | Heap trộn `n-1` dãy tăng | `O((n + k) log n)` |
| 3 | Nhị phân trên đáp án + đếm bằng `bisect` | `O(n log n · log W)` |
| 4 | Nhị phân trên đáp án + đếm bằng hai con trỏ | `O(n log n + n log W)` |

### 7.1. Cách 2 — Heap (trộn các dãy đã sắp)

**Giải thích.** Sắp `nums`. Đẩy vào heap phần tử đầu của mỗi dãy `i`: `(nums[i+1] - nums[i], i, i+1)`. Lặp `k` lần: pop `(d, i, j)`; nếu `j + 1 < n` thì đẩy phần tử kế tiếp của dãy `i`: `(nums[j+1] - nums[i], i, j+1)`. Lần pop thứ `k` cho đáp án.

In [9]:
def kth_heap(nums, k):
    a = sorted(nums)
    n = len(a)
    heap = [(a[i + 1] - a[i], i, i + 1) for i in range(n - 1)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        d, i, j = heapq.heappop(heap)
        if j + 1 < n:
            heapq.heappush(heap, (a[j + 1] - a[i], i, j + 1))
    return heap[0][0]

print(kth_heap([1, 3, 1], 1), kth_heap([1, 1, 1], 2), kth_heap([1, 6, 1], 3))

0 0 5


Giải thích:

- `heapify` biến list thành heap trong `O(n)`.
- Pop `k - 1` lần, phần tử nhỏ nhất còn lại (`heap[0]`) là phần tử thứ `k`.
- Mỗi cặp `(i, j)` được đẩy vào heap đúng một lần (từ `(i, j-1)` hoặc ban đầu), nên không trùng.

**Code nộp LeetCode (Cách 2)** — đúng nhưng **TLE** khi `k` lớn:

```python
import heapq

class Solution:
    def smallestDistancePair(self, nums: List[int], k: int) -> int:
        a = sorted(nums); n = len(a)
        heap = [(a[i + 1] - a[i], i, i + 1) for i in range(n - 1)]
        heapq.heapify(heap)
        for _ in range(k - 1):
            d, i, j = heapq.heappop(heap)
            if j + 1 < n:
                heapq.heappush(heap, (a[j + 1] - a[i], i, j + 1))
        return heap[0][0]
```

**Độ phức tạp.** `O(n log n + k log n)` thời gian, `O(n)` bộ nhớ. Với `k ≈ 5·10^7` thì quá chậm.

### 7.2. Cách 3 — Nhị phân trên đáp án + đếm bằng `bisect`

**Giải thích.** `count(d) = Σ_i (bisect_right(a, a[i] + d) - i - 1)`: với mỗi `i`, số `j > i` có `a[j] ≤ a[i] + d`.

In [10]:
def count_bisect(a, d):
    return sum(bisect_right(a, a[i] + d) - i - 1 for i in range(len(a)))

def kth_bsearch_bisect(nums, k):
    a = sorted(nums)
    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_bisect(a, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(kth_bsearch_bisect([1, 3, 1], 1), kth_bsearch_bisect([1, 1, 1], 2), kth_bsearch_bisect([1, 6, 1], 3))

0 0 5


**Code nộp LeetCode (Cách 3):**

```python
from bisect import bisect_right

class Solution:
    def smallestDistancePair(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        lo, hi = 0, a[-1] - a[0]
        while lo < hi:
            mid = (lo + hi) // 2
            cnt = sum(bisect_right(a, a[i] + mid) - i - 1 for i in range(len(a)))
            if cnt >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo
```

**Độ phức tạp.** Mỗi lần đếm `O(n log n)`, `O(log W)` lần → `O(n log n log W)` ≈ `10^4 · 14 · 20 ≈ 3·10^6` — nhanh. Bộ nhớ `O(n)` (hoặc `O(1)` thêm nếu sắp xếp tại chỗ).

### 7.3. Cách 4 — Nhị phân trên đáp án + đếm bằng hai con trỏ

In [11]:
def count_two_pointers(a, d):
    cnt, left = 0, 0
    for right in range(len(a)):
        while a[right] - a[left] > d:
            left += 1
        cnt += right - left              # các cặp (left..right-1, right)
    return cnt

def kth_bsearch_tp(nums, k):
    a = sorted(nums)
    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_two_pointers(a, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(kth_bsearch_tp([1, 3, 1], 1), kth_bsearch_tp([1, 1, 1], 2), kth_bsearch_tp([1, 6, 1], 3))

0 0 5


Giải thích từng dòng của `count_two_pointers`:

- `right` duyệt mọi phần tử làm **đầu phải** của cặp.
- `while a[right] - a[left] > d: left += 1` — đẩy `left` cho tới khi cặp `(left, right)` hợp lệ. Vì `a[right]` tăng dần nên `left` không bao giờ phải lùi.
- `cnt += right - left` — mọi `i` trong `[left, right - 1]` ghép với `right` đều có khoảng cách `≤ d` (vì `a[i] ≥ a[left]`).
- Tổng số bước của `left` ≤ `n` → cả hàm `O(n)`.

**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def smallestDistancePair(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        n = len(a)
        lo, hi = 0, a[-1] - a[0]
        while lo < hi:
            mid = (lo + hi) // 2
            cnt, left = 0, 0
            for right in range(n):
                while a[right] - a[left] > mid:
                    left += 1
                cnt += right - left
            if cnt >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo
```

**Độ phức tạp.** Sắp xếp `O(n log n)` + `O(log W)` lần đếm `O(n)` → `O(n log n + n log W)`. Bộ nhớ `O(n)` cho mảng đã sắp (hoặc `O(1)` thêm).

## 8. TRACE từng bước

### 8.1. Bảng nhị phân cho `nums = [1, 6, 1, 3, 9]`, `k = 6`

Sắp xếp: `a = [1, 1, 3, 6, 9]`; mọi khoảng cách (10 cặp) đã sắp: `[0, 2, 2, 3, 3, 5, 5, 6, 8, 8]` → thứ 6 là **5**.

| Vòng | `lo` | `hi` | `mid` | `count(mid)` | `≥ 6`? | Cập nhật |
|---|---|---|---|---|---|---|
| 1 | 0 | 8 | 4 | 5 | không | `lo = 5` |
| 2 | 5 | 8 | 6 | 8 | có | `hi = 6` |
| 3 | 5 | 6 | 5 | 7 | có | `hi = 5` |
| — | 5 | 5 | | | | dừng, đáp án **5** |

### 8.2. `trace_bsearch`

In [12]:
def trace_bsearch(nums, k):
    a = sorted(nums)
    print(f"a (đã sắp) = {a}, k = {k}")
    print("  mọi khoảng cách:", sorted(a[j] - a[i] for i in range(len(a)) for j in range(i + 1, len(a))))
    lo, hi = 0, a[-1] - a[0]
    step = 1
    while lo < hi:
        mid = (lo + hi) // 2
        c = count_two_pointers(a, mid)
        if c >= k:
            print(f"  vòng {step}: lo={lo}, hi={hi}, mid={mid}, count={c} >= {k} -> hi = {mid}")
            hi = mid
        else:
            print(f"  vòng {step}: lo={lo}, hi={hi}, mid={mid}, count={c} <  {k} -> lo = {mid + 1}")
            lo = mid + 1
        step += 1
    print("  đáp án:", lo)
    return lo

trace_bsearch([1, 6, 1, 3, 9], 6)

a (đã sắp) = [1, 1, 3, 6, 9], k = 6
  mọi khoảng cách: [0, 2, 2, 3, 3, 5, 5, 6, 8, 8]
  vòng 1: lo=0, hi=8, mid=4, count=5 <  6 -> lo = 5
  vòng 2: lo=5, hi=8, mid=6, count=8 >= 6 -> hi = 6
  vòng 3: lo=5, hi=6, mid=5, count=7 >= 6 -> hi = 5
  đáp án: 5


5

### 8.3. `trace_count` — hai con trỏ khi đếm `count(4)`

In [13]:
def trace_count(a, d):
    cnt, left = 0, 0
    print(f"đếm số cặp có khoảng cách <= {d} trong {a}")
    for right in range(len(a)):
        moved = []
        while a[right] - a[left] > d:
            moved.append(left)
            left += 1
        cnt += right - left
        print(f"  right={right} (a={a[right]}): left tiến qua {moved} -> left={left}; "
              f"cộng {right - left} cặp {[(a[i], a[right]) for i in range(left, right)]}; cnt={cnt}")
    return cnt

trace_count([1, 1, 3, 6, 9], 4)

đếm số cặp có khoảng cách <= 4 trong [1, 1, 3, 6, 9]
  right=0 (a=1): left tiến qua [] -> left=0; cộng 0 cặp []; cnt=0
  right=1 (a=1): left tiến qua [] -> left=0; cộng 1 cặp [(1, 1)]; cnt=1
  right=2 (a=3): left tiến qua [] -> left=0; cộng 2 cặp [(1, 3), (1, 3)]; cnt=3
  right=3 (a=6): left tiến qua [0, 1] -> left=2; cộng 1 cặp [(3, 6)]; cnt=4
  right=4 (a=9): left tiến qua [2] -> left=3; cộng 1 cặp [(6, 9)]; cnt=5


5

### 8.4. `trace_heap` — trạng thái heap qua `k` lần lấy

In [14]:
def trace_heap(nums, k):
    a = sorted(nums)
    n = len(a)
    heap = [(a[i + 1] - a[i], i, i + 1) for i in range(n - 1)]
    heapq.heapify(heap)
    print(f"a = {a}; heap ban đầu = {sorted(heap)}")
    for t in range(1, k + 1):
        d, i, j = heapq.heappop(heap)
        if j + 1 < n:
            heapq.heappush(heap, (a[j + 1] - a[i], i, j + 1))
        print(f"  lần {t}: lấy d={d} (cặp {i},{j}); heap còn {sorted(heap)}")
    return d

trace_heap([1, 6, 1, 3, 9], 6)

a = [1, 1, 3, 6, 9]; heap ban đầu = [(0, 0, 1), (2, 1, 2), (3, 2, 3), (3, 3, 4)]
  lần 1: lấy d=0 (cặp 0,1); heap còn [(2, 0, 2), (2, 1, 2), (3, 2, 3), (3, 3, 4)]
  lần 2: lấy d=2 (cặp 0,2); heap còn [(2, 1, 2), (3, 2, 3), (3, 3, 4), (5, 0, 3)]
  lần 3: lấy d=2 (cặp 1,2); heap còn [(3, 2, 3), (3, 3, 4), (5, 0, 3), (5, 1, 3)]
  lần 4: lấy d=3 (cặp 2,3); heap còn [(3, 3, 4), (5, 0, 3), (5, 1, 3), (6, 2, 4)]
  lần 5: lấy d=3 (cặp 3,4); heap còn [(5, 0, 3), (5, 1, 3), (6, 2, 4)]
  lần 6: lấy d=5 (cặp 0,3); heap còn [(5, 1, 3), (6, 2, 4), (8, 0, 4)]


5

## 9. Code nộp LeetCode

Chọn **Cách 4** (nhị phân trên đáp án + hai con trỏ) — `O(n log n + n log W)`.

In [15]:
from typing import List

class Solution:
    def smallestDistancePair(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        n = len(a)
        lo, hi = 0, a[-1] - a[0]
        while lo < hi:
            mid = (lo + hi) // 2
            cnt, left = 0, 0
            for right in range(n):
                while a[right] - a[left] > mid:
                    left += 1
                cnt += right - left
            if cnt >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

assert Solution().smallestDistancePair([1, 3, 1], 1) == 0
assert Solution().smallestDistancePair([1, 1, 1], 2) == 0
assert Solution().smallestDistancePair([1, 6, 1], 3) == 5
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `d_1 ≤ d_2 ≤ … ≤ d_M` là mọi khoảng cách (có lặp), `M = n(n-1)/2`. Đáp án là `d_k`.

**Tính đơn điệu.** `count(d) = #{t : d_t ≤ d}` không giảm theo `d`.

**Đặc trưng.** `count(d_k) ≥ k` (vì `d_1..d_k ≤ d_k`). Với mọi `d < d_k`: chỉ các `d_t ≤ d < d_k` được đếm, mà `t` như vậy phải `< k` → `count(d) ≤ k - 1 < k`. Vậy `d_k` = **`d` nhỏ nhất có `count(d) ≥ k`**.

**Tìm kiếm nhị phân đúng.** Bất biến: đáp án luôn nằm trong `[lo, hi]`. Ban đầu `[0, max - min]` chứa mọi khoảng cách. Nếu `count(mid) ≥ k` thì đáp án `≤ mid` → `hi = mid` giữ bất biến; ngược lại đáp án `> mid` → `lo = mid + 1`. Đoạn co lại mỗi vòng (vì `mid < hi`), dừng khi `lo == hi` = đáp án.

**Đếm đúng (hai con trỏ).** Với mảng đã sắp và `right` cố định, tập `i < right` có `a[right] - a[i] ≤ d` là `{left, …, right - 1}` với `left` nhỏ nhất thoả mãn (vì `a[i]` tăng theo `i`). Khi `right` tăng, `a[right]` tăng nên `left` nhỏ nhất thoả mãn không giảm → không cần lùi `left`. Cộng `right - left` cho mọi `right` đếm mỗi cặp hợp lệ đúng một lần (theo đầu phải).

**Heap (Cách 2)** đúng vì đây là phép trộn `n-1` dãy tăng: phần tử nhỏ nhất chưa lấy luôn nằm ở heap (đầu của một dãy).

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê + sắp xếp | `O(n² log n)` | `O(n²)` |
| 2. Heap | `O((n + k) log n)` | `O(n)` |
| 3. Nhị phân + bisect | `O(n log n · log W)` | `O(n)` |
| 4. Nhị phân + hai con trỏ | `O(n log n + n log W)` | `O(n)` |

(`W = max - min ≤ 10^6` → `log W ≈ 20`.)

In [16]:
import random, time

random.seed(719)
mid_nums = [random.randint(0, 10**6) for _ in range(1500)]
M = len(mid_nums) * (len(mid_nums) - 1) // 2
for k in [1, M // 2, M]:
    row = []
    res = set()
    for name, fn in [("brute", kth_brute), ("heap", kth_heap), ("bisect", kth_bsearch_bisect), ("2ptr", kth_bsearch_tp)]:
        if name == "heap" and k > 200000:
            row.append("heap=bỏ qua (k quá lớn)")
            continue
        t0 = time.perf_counter()
        res.add(fn(mid_nums, k))
        row.append(f"{name}={time.perf_counter() - t0:.3f}s")
    assert len(res) == 1
    print(f"n=1500, k={k:>7}: đáp án {res.pop():>6} | " + ", ".join(row))

big = [random.randint(0, 10**6) for _ in range(10**4)]
Kbig = 10**4 * (10**4 - 1) // 4
for name, fn in [("bisect", kth_bsearch_bisect), ("2ptr", kth_bsearch_tp)]:
    t0 = time.perf_counter()
    r = fn(big, Kbig)
    print(f"n=10^4, k={Kbig}: {name} -> {r}  {time.perf_counter() - t0:.3f}s")

n=1500, k=      1: đáp án      0 | brute=0.327s, heap=0.001s, bisect=0.004s, 2ptr=0.002s


n=1500, k= 562125: đáp án 299425 | brute=0.310s, heap=bỏ qua (k quá lớn), bisect=0.004s, 2ptr=0.002s


n=1500, k=1124250: đáp án 999357 | brute=0.284s, heap=bỏ qua (k quá lớn), bisect=0.003s, 2ptr=0.002s
n=10^4, k=24997500: bisect -> 294299  0.033s
n=10^4, k=24997500: 2ptr -> 294299  0.018s


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `n = 2`; mọi phần tử bằng nhau; `k = 1` (thường là 0 nếu có trùng, hoặc hiệu kề nhỏ nhất); `k = M` (lớn nhất = `max - min`); mảng chưa sắp xếp; có số 0 và `10^6`.
3. Ngẫu nhiên có seed: mọi cách so với brute force, **với mọi `k`** từ 1 tới `M`.

In [17]:
METHODS = [kth_heap, kth_bsearch_bisect, kth_bsearch_tp, lambda a, k: Solution().smallestDistancePair(a, k)]

def check(nums, k, expected=None):
    ref = kth_brute(nums, k)
    for f in METHODS:
        got = f(nums[:], k)
        assert got == ref, (nums, k, got, ref)
    if expected is not None:
        assert ref == expected, (nums, k, ref, expected)
    return ref

check([1, 3, 1], 1, 0)
check([1, 1, 1], 2, 0)
check([1, 6, 1], 3, 5)
check([5, 9], 1, 4)                          # n = 2
check([7, 7, 7, 7], 6, 0)                    # mọi phần tử bằng nhau
check([1, 4, 10, 20], 1, 3)                  # k = 1 -> hiệu kề nhỏ nhất
check([1, 4, 10, 20], 6, 19)                 # k = M -> max - min
check([0, 1000000], 1, 1000000)
check([9, 1, 6, 3, 1], 6, 5)                 # chưa sắp xếp
print("Ví dụ + edge case: OK")

random.seed(2032)
total = 0
for _ in range(300):
    n = random.randint(2, 9)
    nums = [random.randint(0, random.choice([3, 10, 100])) for _ in range(n)]
    for k in range(1, n * (n - 1) // 2 + 1):
        check(nums, k)
        total += 1
print(f"{total} bộ (nums, k) ngẫu nhiên: mọi cách khớp brute force")

Ví dụ + edge case: OK
4677 bộ (nums, k) ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Tìm `mid` có `count(mid) == k` (đúng bằng) rồi trả về

In [18]:
def kth_wrong_exact(nums, k):
    a = sorted(nums)
    lo, hi = 0, a[-1] - a[0]
    while lo <= hi:
        mid = (lo + hi) // 2
        c = count_two_pointers(a, mid)
        if c == k:                                # SAI: count có thể "nhảy qua" k
            return mid
        if c < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print("sai:", kth_wrong_exact([1, 1, 1], 2), "| đúng:", kth_brute([1, 1, 1], 2))

sai: -1 | đúng: 0


**Vì sao sai:** nhiều cặp cùng khoảng cách làm `count` nhảy từ dưới `k` lên trên `k` (ở đây `count(0) = 3`, không có `d` nào cho `count = 2`). Phải tìm `d` **nhỏ nhất** có `count(d) ≥ k`.

### Lỗi 2: Quên sắp xếp trước khi dùng hai con trỏ

In [19]:
def kth_unsorted(nums, k):
    a = nums[:]                                    # SAI: không sắp xếp
    lo, hi = 0, max(a) - min(a)
    while lo < hi:
        mid = (lo + hi) // 2
        cnt, left = 0, 0
        for right in range(len(a)):
            while left < right and a[right] - a[left] > mid:
                left += 1
            cnt += right - left
        if cnt >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print("sai:", kth_unsorted([5, 1, 3], 1), "| đúng:", kth_brute([5, 1, 3], 1))

sai: 0 | đúng: 2


**Vì sao sai:** hai con trỏ dựa vào việc `a[right] - a[i]` giảm khi `i` tăng (trên mảng đã sắp). Mảng chưa sắp thì cặp hợp lệ không còn là một đoạn liên tiếp, và hiệu có thể âm. Luôn `a = sorted(nums)`.

### Lỗi 3: Đếm cả cặp `(right, right)` — cộng `right - left + 1`

In [20]:
def count_with_self(a, d):
    cnt, left = 0, 0
    for right in range(len(a)):
        while a[right] - a[left] > d:
            left += 1
        cnt += right - left + 1                    # SAI: tính cả cặp chính nó
    return cnt

def kth_with_self(nums, k):
    a = sorted(nums)
    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_with_self(a, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print("sai:", kth_with_self([1, 6, 1, 3, 9], 6), "| đúng:", kth_brute([1, 6, 1, 3, 9], 6))

sai: 0 | đúng: 5


**Vì sao sai:** cặp phải có `i < j`; cộng thêm 1 ở mỗi `right` làm `count` bị thừa `n` → đáp án nhỏ hơn thực tế. Đúng: `cnt += right - left`.

### Lỗi 4: Dùng `lo <= hi` với `hi = mid` → vòng lặp vô hạn

Mẫu "giá trị nhỏ nhất thoả mãn" dùng `while lo < hi` cùng `hi = mid` / `lo = mid + 1`. Nếu đổi thành `lo <= hi` mà vẫn `hi = mid`, khi `lo == hi` và điều kiện đúng thì `hi` không đổi → lặp mãi. Chọn một mẫu và giữ nhất quán.

## 14. Mẹo & nhận dạng pattern

- **"Phần tử nhỏ thứ k" trong một tập ngầm rất lớn** (cặp, tích, phân số, ô trong bảng) → **nhị phân trên giá trị đáp án** + hàm **đếm** số phần tử `≤ x`.
- Hàm đếm trên mảng đã sắp thường làm được bằng **hai con trỏ** `O(n)` hoặc `bisect` `O(n log n)`.
- **"k nhỏ"** → heap trộn các dãy đã sắp cũng là lựa chọn tốt.
- Nhận diện tính đơn điệu: "nếu `x` thoả thì mọi `x' > x` cũng thoả".
- Bài cùng họ: *Kth Smallest Element in a Sorted Matrix* (378), *Kth Smallest Number in Multiplication Table* (668), *K-th Smallest Prime Fraction* (786), *Find K Pairs with Smallest Sums* (373), *Split Array Largest Sum* (410).

## 15. Biến thể

1. **Khoảng cách lớn thứ k** → khoảng cách nhỏ thứ `M - k + 1`.
2. **Đếm số cặp có khoảng cách trong `[lo, hi]`** → `count(hi) - count(lo - 1)`.
3. **LeetCode 668 — số nhỏ thứ k trong bảng nhân `m × n`** → nhị phân trên giá trị, `count(x) = Σ_i min(x // i, n)`.
4. **Cặp từ hai mảng khác nhau** (`|a_i - b_j|`) → sắp cả hai, đếm bằng hai con trỏ trên hai mảng.

Ví dụ biến thể 3 (LeetCode 668):

In [21]:
def kth_in_mult_table(m, n, k):
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        cnt = sum(min(mid // i, n) for i in range(1, m + 1))
        if cnt >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def kth_mult_brute(m, n, k):
    return sorted(i * j for i in range(1, m + 1) for j in range(1, n + 1))[k - 1]

print(kth_in_mult_table(3, 3, 5), kth_in_mult_table(2, 3, 6))   # 3, 6
random.seed(668)
for _ in range(300):
    m, n = random.randint(1, 8), random.randint(1, 8)
    k = random.randint(1, m * n)
    assert kth_in_mult_table(m, n, k) == kth_mult_brute(m, n, k)
print("biến thể 668 khớp brute force")

3 6
biến thể 668 khớp brute force


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê + sắp xếp | Mọi khoảng cách, lấy phần tử `k-1` | `O(n² log n)` | `O(n²)` | Trọng tài khi test, `n` nhỏ |
| 2. Heap | Trộn `n-1` dãy tăng, pop `k` lần | `O((n + k) log n)` | `O(n)` | `k` nhỏ |
| 3. Nhị phân + bisect | `count(d)` bằng `bisect_right` | `O(n log n log W)` | `O(n)` | Dễ viết, đủ nhanh |
| 4. Nhị phân + hai con trỏ | `count(d)` trong `O(n)` | `O(n log n + n log W)` | `O(n)` | Tối ưu (khuyên dùng) |

## 17. Bài tập

**Bài 1 (đoán output).** `smallestDistancePair([1, 2, 3, 4], 3)`, `([10, 10, 20], 2)`, `([4, 1], 1)`?

**Bài 2 (tính tay).** Lập bảng nhị phân (như 8.1) cho `nums = [2, 8, 3, 3]`, `k = 4`.

**Bài 3 (điền TODO).** Hoàn thiện `my_count(a, d)` bằng hai con trỏ.

**Bài 4 (tự viết lại).** Viết lại `my_kth(nums, k)` hoàn chỉnh (sắp xếp + nhị phân + đếm).

**Bài 5 (mini challenge).** Viết `kth_cross(A, B, k)`: khoảng cách nhỏ thứ `k` trong mọi `|A[i] - B[j]|` (mọi cặp `i`, `j` giữa hai mảng). Gợi ý: sắp cả hai mảng; `count(d)` = với mỗi `B[j]`, số `A[i]` trong `[B[j] - d, B[j] + d]` (dùng `bisect`).

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_count(a, d):
    # TODO: cnt, left = 0, 0; với right: đẩy left khi a[right] - a[left] > d; cnt += right - left
    pass

# Bài 4 — TODO
def my_kth(nums, k):
    # TODO: a = sorted(nums); lo, hi = 0, a[-1] - a[0]; while lo < hi: ... ; return lo
    pass

# Bài 5 — TODO
def kth_cross(A, B, k):
    # TODO: sắp A, B; lo = 0, hi = max khoảng cách; count(d) = Σ_j bisect_right(A, B[j]+d) - bisect_left(A, B[j]-d)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê khoảng cách.
- Mức 2: `[1,2,3,4]`: `1,1,1,2,2,3`.
- Mức 3: `1`; `[10,10,20]`: `0,10,10` → `10`; `[4,1]`: `3`.

**Bài 2.**
- Mức 1: sắp xếp `[2, 3, 3, 8]`; khoảng cách đã sắp: `0, 1, 1, 5, 5, 6`.
- Mức 2: `lo=0, hi=6`: `mid=3`, `count=3 < 4` → `lo=4`; `mid=5`, `count=5 ≥ 4` → `hi=5`; `mid=4`, `count=3` → `lo=5`.
- Mức 3: đáp án `5`.

**Bài 3.**
- Mức 1: `left` chỉ tiến, không lùi.
- Mức 2: điều kiện `while a[right] - a[left] > d`.
- Mức 3: cộng `right - left` (không `+1`).

**Bài 4.**
- Mức 1: nhớ sắp xếp.
- Mức 2: mẫu `while lo < hi`, `hi = mid` khi `count ≥ k`.
- Mức 3: trả `lo`.

**Bài 5.**
- Mức 1: `hi = max(|A[0] - B[-1]|, |A[-1] - B[0]|)`.
- Mức 2: với mỗi `b`, số `a` thoả `|a - b| ≤ d` là `bisect_right(A, b + d) - bisect_left(A, b - d)`.
- Mức 3: cùng mẫu nhị phân "nhỏ nhất có count ≥ k".

## 19. Đáp án tham khảo

In [23]:
from bisect import bisect_left

S = Solution()
# Bài 1
assert S.smallestDistancePair([1, 2, 3, 4], 3) == 1
assert S.smallestDistancePair([10, 10, 20], 2) == 10
assert S.smallestDistancePair([4, 1], 1) == 3

# Bài 2
a = sorted([2, 8, 3, 3])
assert [count_two_pointers(a, d) for d in (3, 5, 4)] == [3, 5, 3]
assert S.smallestDistancePair([2, 8, 3, 3], 4) == 5

# Bài 3
def my_count_ans(a, d):
    cnt, left = 0, 0
    for right in range(len(a)):
        while a[right] - a[left] > d:
            left += 1
        cnt += right - left
    return cnt

assert my_count_ans([1, 1, 3, 6, 9], 4) == 5
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_kth_ans(nums, k):
    a = sorted(nums)
    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if my_count_ans(a, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

# Bài 5
def kth_cross_ans(A, B, k):
    A, B = sorted(A), sorted(B)
    lo, hi = 0, max(abs(A[0] - B[-1]), abs(A[-1] - B[0]))
    while lo < hi:
        mid = (lo + hi) // 2
        cnt = sum(bisect_right(A, b + mid) - bisect_left(A, b - mid) for b in B)
        if cnt >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def kth_cross_brute(A, B, k):
    return sorted(abs(x - y) for x in A for y in B)[k - 1]

random.seed(73)
for _ in range(400):
    nums = [random.randint(0, 20) for _ in range(random.randint(2, 8))]
    k = random.randint(1, len(nums) * (len(nums) - 1) // 2)
    assert my_kth_ans(nums, k) == kth_brute(nums, k)
    A = [random.randint(0, 20) for _ in range(random.randint(1, 6))]
    B = [random.randint(0, 20) for _ in range(random.randint(1, 6))]
    k2 = random.randint(1, len(A) * len(B))
    assert kth_cross_ans(A, B, k2) == kth_cross_brute(A, B, k2)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "phần tử nhỏ thứ k" bằng nhị phân trên đáp án:**

```python
a = sorted(nums)
lo, hi = 0, a[-1] - a[0]
while lo < hi:
    mid = (lo + hi) // 2
    if count(mid) >= k: hi = mid      # count(d) = số cặp có khoảng cách <= d
    else:               lo = mid + 1
return lo
```

**Template đếm bằng hai con trỏ:**

```python
cnt = left = 0
for right in range(n):
    while a[right] - a[left] > d: left += 1
    cnt += right - left
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** `nums`.
- [ ] Tìm `d` **nhỏ nhất** có `count(d) ≥ k` (không tìm `count == k`).
- [ ] `hi = a[-1] - a[0]` (khoảng cách lớn nhất), `lo = 0`.
- [ ] Đếm cặp `i < j`: `cnt += right - left` (không `+1`).
- [ ] Mẫu nhị phân nhất quán: `while lo < hi`, `hi = mid`, `lo = mid + 1`.
- [ ] Test `[1,3,1],1 → 0`; `[1,1,1],2 → 0`; `[1,6,1],3 → 5`.
- [ ] Tên method đúng: `smallestDistancePair(self, nums, k)`.